# Lecture 4: Feature Selection, Scaling and LassoCV

## Algerian Forest Fires project

Goal: predict **FWI (Fire Weather Index)** from weather and fire measurements. This notebook turns the model-training transcript into a short, safe, repeatable workflow.

**Road map:** clean labels → split data → find duplicate information → scale safely → compare Linear Regression, Lasso and LassoCV.

## Why prepare data before training?

- **Feature selection** removes columns that tell almost the same story. This can reduce multicollinearity and make a model easier to understand.
- **Scaling** puts numeric columns on a similar range. Lasso uses coefficient size, so it needs scaling.
- **Cross-validation** tries several train/validation splits inside the training data to choose a good Lasso strength.

Important rule: learn every decision from the **training set only**. The test set stays untouched until the final check.

In [ ]:
# Cell 1: imports and data loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
df = pd.read_csv('Model Training Practicals/Algerian_forest_fires_cleaned_dataset.csv')
df.head()

### What changed here?

We load the cleaned CSV and import the tools used later. pandas stores the table, seaborn and matplotlib draw visuals, and scikit-learn builds models.

In [ ]:
# Cell 2: clean the class label and choose X (inputs) and y (answer)
df['Classes'] = (df['Classes'].astype(str).str.strip().str.lower()
                 .map({'not fire': 0, 'fire': 1}))
target = 'FWI'
X = df.drop(columns=['day', 'month', 'year', target])
y = df[target]
print('Input shape:', X.shape)
print('Target:', target)
print('Missing values:', X.isna().sum().sum())
X.head()

### Why drop day, month and year?

The transcript treats them as calendar identifiers rather than useful weather measurements for this example. We keep **FWI** out of X because it is the value we want to predict. Classes changes from words to 0/1, and strip removes hidden spaces before mapping.

In [ ]:
# Cell 3: split before feature selection or scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print('Training rows:', X_train.shape[0])
print('Test rows:', X_test.shape[0])
plt.figure(figsize=(7, 3))
plt.bar(['Training data', 'Test data'], [len(X_train), len(X_test)], color=['#4C78A8', '#F58518'])
plt.title('Keep the test set for the final exam')
plt.ylabel('Number of rows')
plt.show()

## Visual 1: find features that overlap

Correlation is from -1 to +1. A value near +1 means two features rise together; near -1 means one rises while the other falls. Both are strong relationships, so we use **absolute correlation**.

In [ ]:
# Cell 4: correlation heatmap using TRAINING data only
plt.figure(figsize=(10, 7))
sns.heatmap(X_train.corr(), cmap='coolwarm', center=0, linewidths=0.4)
plt.title('Feature correlation map (training data only)')
plt.show()

In [ ]:
# Cell 5: remove one feature from each highly correlated pair
def highly_correlated_features(data, threshold=0.85):
    correlation = data.corr().abs()
    upper_triangle = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
    return [column for column in upper_triangle.columns if (upper_triangle[column] > threshold).any()]

correlated_features = highly_correlated_features(X_train, threshold=0.85)
X_train_reduced = X_train.drop(columns=correlated_features)
X_test_reduced = X_test.drop(columns=correlated_features)
print('Dropped because of high correlation:', correlated_features)
print('Features before:', X_train.shape[1], '| Features after:', X_train_reduced.shape[1])

### Understand the feature-selection edit

The upper triangle prevents checking the same pair twice. When a pair is above 0.85, this simple rule drops one column. The exact threshold is **not universal**: use domain knowledge and validation results too. We apply the training-set decision to X_test_reduced; we never calculate new test-set correlations.

In [ ]:
# Cell 6: a visual of why scaling matters
scaler_demo = StandardScaler()
scaled_train_demo = pd.DataFrame(scaler_demo.fit_transform(X_train_reduced), columns=X_train_reduced.columns)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=X_train_reduced, ax=axes[0], color='#72B7B2')
axes[0].set_title('Before scaling: different units')
axes[0].tick_params(axis='x', rotation=70)
sns.boxplot(data=scaled_train_demo, ax=axes[1], color='#ECA82C')
axes[1].set_title('After StandardScaler: comparable scale')
axes[1].tick_params(axis='x', rotation=70)
plt.tight_layout()
plt.show()

## The safe scaling pattern

Normally we would fit a scaler on X_train and then transform both train and test. Below, a **pipeline** does this automatically. It is even safer for cross-validation: each fold learns its scaler only from that fold's training portion. This improves the transcript's manual scaling approach by avoiding accidental validation leakage.

In [ ]:
# Cell 7: compare a baseline, a chosen Lasso, and LassoCV
linear_model = make_pipeline(StandardScaler(), LinearRegression())
lasso_model = make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=20000))
lasso_cv_model = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=20000, random_state=42))
models = {'Linear Regression': linear_model, 'Lasso (alpha=0.1)': lasso_model, 'LassoCV (5 folds)': lasso_cv_model}
results = []
for name, model in models.items():
    model.fit(X_train_reduced, y_train)
    prediction = model.predict(X_test_reduced)
    results.append([name, mean_absolute_error(y_test, prediction), r2_score(y_test, prediction)])
results_df = pd.DataFrame(results, columns=['Model', 'Test MAE (lower is better)', 'Test R2 (higher is better)'])
results_df

In [ ]:
# Cell 8: see how LassoCV chooses alpha
chosen_lasso = lasso_cv_model.named_steps['lassocv']
mean_mse = chosen_lasso.mse_path_.mean(axis=1)
plt.figure(figsize=(8, 4))
plt.semilogx(chosen_lasso.alphas_, mean_mse, marker='o', ms=3, label='Mean validation MSE')
plt.axvline(chosen_lasso.alpha_, color='crimson', linestyle='--', label=f'Chosen alpha = {chosen_lasso.alpha_:.4f}')
plt.gca().invert_xaxis()
plt.xlabel('Alpha (regularization strength)')
plt.ylabel('Mean validation MSE')
plt.title('LassoCV selects the alpha with the best validation error')
plt.legend()
plt.show()
print('Best alpha selected by 5-fold CV:', chosen_lasso.alpha_)

### How to read the diagram

- Small alpha: Lasso behaves more like ordinary Linear Regression.
- Large alpha: Lasso penalizes coefficients harder and may set some to zero.
- LassoCV chooses the alpha with the lowest average validation error, then we report performance once on the untouched test set.

In [ ]:
# Cell 9: final prediction visual for the selected LassoCV model
final_prediction = lasso_cv_model.predict(X_test_reduced)
lower = min(y_test.min(), final_prediction.min())
upper = max(y_test.max(), final_prediction.max())
plt.figure(figsize=(6, 5))
plt.scatter(y_test, final_prediction, color='#4C78A8', alpha=0.8)
plt.plot([lower, upper], [lower, upper], '--', color='crimson', label='Perfect prediction')
plt.xlabel('Actual FWI')
plt.ylabel('Predicted FWI')
plt.title('LassoCV: actual vs predicted FWI')
plt.legend()
plt.show()

## Quick revision card

1. Put the prediction target in y; keep possible inputs in X.
2. Split first. Make feature-selection and scaling decisions with training data.
3. Use absolute correlation to spot duplicated information.
4. Scale for Lasso because its penalty depends on coefficient size.
5. LassoCV with five folds tests alpha values through repeated training/validation splits.
6. Use the test set only for the final MAE and R².

**One-line interview answer:** Lasso is linear regression with an L1 penalty; it can shrink weak feature coefficients exactly to zero. LassoCV chooses its penalty strength using cross-validation.